## Generación y selección de características a utilizar


**Características de salida:**
- Transaction_Type_POS, Transaction_Type_Onlin` — one-hot encoder para el tipo de transacción
- Is_Chip — variable binaria
- month — mes extraído de la fecha, escalado con StandardScaler
- Device_Type_Terminal, Device_Type_Web — one-hot encoder de tipo de dispositivo
- Card_Type_Platinum, Card_Type_Debit, Card_Type_Gold — one-hot de tipo de tarjeta
- Is_Pin_Used — variable binaria
- Fraud_Flag — variable objetivo

In [64]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

pd.set_option('display.max_columns', None)

In [65]:
df = pd.read_csv("credit_card_fraud_2025.csv")

# Convertir fecha a datetime
df['Transaction_Date'] = pd.to_datetime(df['Transaction_Date'])

# Eliminar columnas identificadoras que no aportan al modelo
df = df.drop(columns=["Transaction_ID", "Customer_ID", "Merchant_ID"])

print("Shape inicial:", df.shape)
df.head()

Shape inicial: (500000, 13)


,Transaction_Date,Amount,Merchant_Category,Card_Type,Transaction_Type,Country,Is_International,Is_Chip,Is_Pin_Used,Distance_From_Home,Hour_of_Day,Device_Type,Fraud_Flag
0,2025-05-28 11:54:36,81.53,Online Services,Gold,POS,Germany,1,1,0,1.61,11,Web,0
1,2024-09-11 20:26:12,52.19,Fuel,Debit,ATM,Germany,0,1,0,15.77,20,Web,0
2,2024-11-02 12:39:23,27.70,Utilities,Gold,Online,USA,0,1,0,9.19,12,Terminal,0
3,2024-10-08 21:58:08,9.80,Clothing,Gold,ATM,USA,0,1,0,9.42,21,Web,0
4,2024-05-25 20:01:21,178.06,Electronics,Gold,ATM,Germany,0,1,1,1.32,20,Web,0


In [66]:
# Extraer mes — variable temporal con mayor correlación relativa con Fraud_Flag
df['month'] = df['Transaction_Date'].dt.month

# Eliminar la columna de fecha original
df = df.drop(columns=['Transaction_Date'])

print("Valores únicos de month:", sorted(df['month'].unique()))

Valores únicos de month: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]


In [67]:
scaler = StandardScaler()

df['month'] = scaler.fit_transform(df[['month']])

print("month — media:", df['month'].mean().round(6), "| std:", df['month'].std().round(6))

month — media: -0.0 | std: 1.000001


One-hot encoding de variables categóricas

Se aplica a Transaction_Type, Device_Type y Card_Type.
Esto genera columnas de tipo float64.

Las variables binarias Is_Chip e Is_Pin_Used se dejan como int64.

In [68]:
df = pd.get_dummies(df, columns=['Transaction_Type', 'Device_Type', 'Card_Type'], drop_first=False, dtype=float)

# Verificar columnas generadas
ohe_cols = [c for c in df.columns if any(c.startswith(p) for p in ['Transaction_Type_', 'Device_Type_', 'Card_Type_'])]
print("Columnas one-hot generadas:", ohe_cols)

Columnas one-hot generadas: ['Transaction_Type_ATM', 'Transaction_Type_Online', 'Transaction_Type_POS', 'Device_Type_Mobile', 'Device_Type_Terminal', 'Device_Type_Web', 'Card_Type_Credit', 'Card_Type_Debit', 'Card_Type_Gold', 'Card_Type_Platinum']


In [69]:
selected_features = [
    'Transaction_Type_POS',
    'Transaction_Type_Online',
    'Is_Chip',          # int64 — binaria original
    'month',            # float64 — escalada
    'Device_Type_Terminal',
    'Device_Type_Web',
    'Card_Type_Platinum',
    'Is_Pin_Used',      # int64 — binaria original
    'Card_Type_Debit',
    'Card_Type_Gold',
    'Fraud_Flag'
]

# Verificar que todas las columnas existen
missing = [f for f in selected_features if f not in df.columns]

df_final = df[selected_features].copy()

print("\nShape final:", df_final.shape)
df_final.head()


Shape final: (500000, 11)


,Transaction_Type_POS,Transaction_Type_Online,Is_Chip,month,Device_Type_Terminal,Device_Type_Web,Card_Type_Platinum,Is_Pin_Used,Card_Type_Debit,Card_Type_Gold,Fraud_Flag
0,1.0,0.0,1,-0.274963,0.0,1.0,0.0,0,0.0,1.0,0
1,0.0,0.0,1,0.977625,0.0,1.0,0.0,0,1.0,0.0,0
2,0.0,1.0,1,1.603919,1.0,0.0,0.0,0,0.0,1.0,0
3,0.0,0.0,1,1.290772,0.0,1.0,0.0,0,0.0,1.0,0
4,0.0,0.0,1,-0.274963,0.0,1.0,0.0,1,0.0,1.0,0


In [70]:
print("Tipos de datos:")
print(df_final.dtypes)

print("\nValores nulos:")
print(df_final.isnull().sum())

print("\nEstadísticas descriptivas:")
df_final.describe()

Tipos de datos:
Transaction_Type_POS       float64
Transaction_Type_Online    float64
Is_Chip                      int64
month                      float64
Device_Type_Terminal       float64
Device_Type_Web            float64
Card_Type_Platinum         float64
Is_Pin_Used                  int64
Card_Type_Debit            float64
Card_Type_Gold             float64
Fraud_Flag                   int64
dtype: object

Valores nulos:
Transaction_Type_POS       0
Transaction_Type_Online    0
Is_Chip                    0
month                      0
Device_Type_Terminal       0
Device_Type_Web            0
Card_Type_Platinum         0
Is_Pin_Used                0
Card_Type_Debit            0
Card_Type_Gold             0
Fraud_Flag                 0
dtype: int64

Estadísticas descriptivas:


,Transaction_Type_POS,Transaction_Type_Online,Is_Chip,month,Device_Type_Terminal,Device_Type_Web,Card_Type_Platinum,Is_Pin_Used,Card_Type_Debit,Card_Type_Gold,Fraud_Flag
count,500000.000000,500000.000000,500000.000000,5.000000e+05,500000.000000,500000.000000,500000.000000,500000.00000,500000.000000,500000.000000,500000.000000
mean,0.333236,0.333654,0.300222,-5.979217e-17,0.333464,0.332076,0.250650,0.19980,0.250040,0.249508,0.015000
std,0.471371,0.471518,0.458355,1.000001e+00,0.471451,0.470959,0.433388,0.39985,0.433036,0.432729,0.121553
min,0.000000,0.000000,0.000000,-1.527551e+00,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,-9.012572e-01,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000,3.818390e-02,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000
75%,1.000000,1.000000,1.000000,6.644780e-01,1.000000,1.000000,1.000000,0.00000,1.000000,0.000000,0.000000
max,1.000000,1.000000,1.000000,1.917066e+00,1.000000,1.000000,1.000000,1.00000,1.000000,1.000000,1.000000


In [71]:
bool_cols = df_final.select_dtypes(include='bool').columns.tolist()
if bool_cols:
    df_final[bool_cols] = df_final[bool_cols].astype(float)
    print("Columnas bool convertidas a float:", bool_cols)
else:
    print("Sin columnas bool que convertir")

df_final.dtypes

Sin columnas bool que convertir


Transaction_Type_POS       float64
Transaction_Type_Online    float64
Is_Chip                      int64
month                      float64
Device_Type_Terminal       float64
Device_Type_Web            float64
Card_Type_Platinum         float64
Is_Pin_Used                  int64
Card_Type_Debit            float64
Card_Type_Gold             float64
Fraud_Flag                   int64
dtype: object

### 8. Exportación de CSV con características generadas

In [72]:
output_path = "credit_card_fraud_selected_features.csv"
df_final.to_csv(output_path, index=False)

print(f"CSV guardado: {output_path}")
print(f"   Filas: {df_final.shape[0]:,} | Columnas: {df_final.shape[1]}")


CSV guardado: credit_card_fraud_selected_features.csv
   Filas: 500,000 | Columnas: 11


In [73]:
X = df_final.drop(columns=['Fraud_Flag'])
y = df_final['Fraud_Flag']

print("X shape:", X.shape)
print("y shape:", y.shape)
print("\nDistribución de clases:")
print(y.value_counts(normalize=True).rename({0: 'No Fraude', 1: 'Fraude'}))

X shape: (500000, 10)
y shape: (500000,)

Distribución de clases:
Fraud_Flag
No Fraude    0.985
Fraude       0.015
Name: proportion, dtype: float64


Las características seleccionadas para el modelo de detección de fraude fueron elegidas por su capacidad de representar el contexto y la seguridad de cada transacción, permitiendo identificar patrones anómalos.El tipo de transacción es relevante porque las operaciones en línea suelen implicar mayor riesgo al no requerir presencia física; el uso de chip y de PIN aportan información sobre el nivel de autenticación y seguridad, siendo su ausencia un posible indicador de vulnerabilidad; el mes permite capturar patrones temporales o estacionales en la ocurrencia de fraude; el tipo de dispositivo ayuda a detectar cambios o comportamientos inusuales en los canales de uso; y el tipo de tarjeta si fue de débito, credito entre otros lo cual refleja distintos perfiles de consumo y exposición al riesgo. En conjunto, estas variables permiten modelar tanto el comportamiento habitual del usuario como posibles desviaciones, mejorando la capacidad del sistema para detectar transacciones fraudulentas.